# Mapeo de códigos

## Inicializa Spark

In [0]:
%run "./00_Init_Spark"

## Inicializa las funciones

In [0]:
%run "./00_Init_Funcs"

## Lee tablas de códigos

In [0]:
df_ct = spark.read.table("bronze.codigos_territorios")

In [0]:
df_cte = spark.read.table("bronze.codigos_territorios_especificos")

In [0]:
df_co = spark.read.table("bronze.codigos_otros")

## Cruza tablas de entrada con Códigos Territoriales

### Cruce entre Vivienda y Códigos Territoriales

In [0]:
df_vivienda = spark.read.table("bronze.viviendas")

In [0]:
df_vivienda_ct = cruce_codigos_territoriales(df_vivienda, df_ct)

### Cruce entre Hogar y Códigos Territoriales

In [0]:
df_hogar = spark.read.table("bronze.hogares")

In [0]:
df_hogar_ct = cruce_codigos_territoriales(df_hogar, df_ct)

### Cruce entre Persona y Códigos Territoriales

In [0]:
df_persona = spark.read.table("bronze.personas")

In [0]:
df_persona_ct = cruce_codigos_territoriales(df_persona, df_ct)

### Cruce entre Persona y Códigos Territoriales específicos

In [0]:
# Campos de codigos territoriales especificos
campos_cte = [
    "p25_lug_nacimiento_esp",
    "p27_nacionalidad_esp",
    "p44_lug_trab_esp",
    "p24_lug_resid5_esp"]

for c_cte in campos_cte:
    df_persona_ct = df_persona_ct \
        .alias("df_pct") \
        .join(broadcast(df_cte.alias(f"df_cte")), col(f"df_pct.{c_cte}") == col(f"df_cte.codigo_especifico"), "left") \
        .withColumn(c_cte, 
                    when(col("df_cte.codigo_especifico") > 0, col("df_cte.territorio_especifico")) \
                    .when(col(f"df_pct.{c_cte}") == "-66", lit(None)) \
                    .when(col(f"df_pct.{c_cte}") == "-99", lit(None))
                    .otherwise(c_cte)) \
        .drop("codigo_especifico", "territorio_especifico")


## Cruce tablas de entrada con Códigos Varios

### Cruce entre Vivienda y Códigos Varios

In [0]:
df_vivienda_co = cruce_codigos_otros(df_vivienda_ct, df_co, ["id_vivienda"])
display(df_vivienda_co)

### Cruce entre Hogar y Códigos Varios

In [0]:
df_hogar_co = cruce_codigos_otros(df_hogar_ct, df_co, ["id_vivienda", "id_hogar"])
display(df_hogar_co)

### Cruce entre Persona y Códigos Varios

In [0]:
df_persona_co = cruce_codigos_otros(df_persona_ct, df_co, ["id_vivienda", "id_hogar", "id_persona"])
display(df_persona_co)

## Guarda el resultado en las tablas

In [0]:
df_vivienda_co.write.insertInto("silver.viviendas", overwrite=True)

In [0]:
df_hogar_co.write.insertInto("silver.hogares", overwrite=True)

In [0]:

df_persona_co.write.insertInto("silver.personas", overwrite=True)